# TD法

TD法では、更新式が長期報酬の見積もりにどう効くかを、数式とコードの両方で確認します。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=Mv62VMRczUo)
@[youtube](https://www.youtube.com/watch?v=Mv62VMRczUo)

## 参考リンク
- https://www.youtube.com/watch?v=Mv62VMRczUo


## 背景と目的

TD法はモンテカルロ法と動的計画法の中間に位置し、逐次更新で学習できます。

ブートストラップ更新を理解すると、オンライン学習系アルゴリズムの基礎が固まります。

TD誤差が更新方向をどう決めるかを追跡します。


## 最初に解きたい疑問

1. TD法は、モンテカルロ法と何が違うのか。
2. `TD誤差` は、どの値とどの値の差なのか。
3. なぜ先に `return` を計算してから、TD更新を見るのか。
4. この章で Q学習や探索が出るのは、TD法の何を補っているのか。
5. `alpha` を変えると、更新の見え方はどう変わるのか。


## 先に押さえる言葉

- `TD error`: 今の予測と、1ステップ先を使った予測との差です。学習の方向と大きさを決めます。
- `bootstrapping`: 次の推定値を使って、今の推定値を更新することです。完全な未来を待たずに学習できます。
- `learning rate`: 1回の更新で、どれだけ強く値を動かすかを決める係数です。大きすぎると不安定になります。
- `Monte Carlo method`: エピソードの最後まで見てから、実際の総報酬で価値を更新する方法です。逐次更新ではありません。


## 実行前の見取り図

1. `mc_return=` が、終端まで足し切った参照値として出ているか。
2. `updated V(s0)=` の更新後値が、TD誤差で動いているか。
3. `choose_action(...)` の出力で、探索と活用の切り替えが見えるか。


## 出力の読み方

- `mode=explore` ならランダム行動、`mode=greedy` なら現在の価値推定で一番良い行動を選んでいます。
- この探索セルは TD(0) の更新式そのものではなく、価値を集めるために一緒に使う行動選択の最小例です。


## つまずきやすい点

- TD法が『次の推定値で学ぶ』という直感を、数式とコードを対応づけて説明する補足がない。
- Q学習の例が、TD法のどの部分を示しているのかの説明が薄い。
- `TD(0)` の更新と、探索方策の話が同じ章にある理由がまだ見えにくい。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- この探索セルは TD 更新の補助で、探索戦略そのものの学習ではない。


## 手を動かす 1: Monte Carlo 参照値を作る

TD誤差を観察する前に、終端まで足し切った割引収益を参照用に作ります。これは bootstrapping ではなく、Monte Carlo return です。


In [ ]:
rewards = [0.3, 0.0, 0.7, 1.0]
gamma = 0.87
g = 0.0
for r in reversed(rewards):
    g = r + gamma * g
print('task = td-learning', 'mc_return=', round(g, 6))

この値は終端までの実報酬を全部使った Monte Carlo return です。次のセルではこれを参照値として置き、TD(0) が `r + gamma * V(s')` という bootstrapped target で近似的に学ぶ様子を見ます。



## 手を動かす 2: TD(0) の状態価値更新を1回行う

次に、TD(0) で状態価値を 1 ステップだけ更新します。まずは `max` を使わない評価側の更新を見て、TD 法の中心であるブートストラップをはっきり区別します。


In [ ]:
alpha = 0.2
V = {'s0': 0.3, 's1': 0.8}
r, s, s_next = 1.0, 's0', 's1'
td_target = r + gamma * V[s_next]
td_error = td_target - V[s]
V[s] = V[s] + alpha * td_error
print('TD target =', round(td_target, 6))
print('TD error =', round(td_error, 6))
print('updated V(s0)=', round(V[s], 6))


TD(0) では、`r + gamma * V(s')` を 1 ステップ先の目標値として使い、今の `V(s)` を少しだけ寄せます。これが TD 法の基本で、次のセルで Q 学習との差分として `max` を導入します。


## 数式メモ

1. $G_t = \sum_{k\ge 0} \gamma^k R_{t+k+1}$
2. $V(s_t) \leftarrow V(s_t) + \alpha[r_{t+1} + \gamma V(s_{t+1}) - V(s_t)]$


## 手を動かす 3: Q学習との差分を見る

ここでは TD の枠組みを Q 学習へ拡張し、状態価値 `V` の更新が行動価値 `Q` と `max` を使う形へどう変わるかを確認します。


In [ ]:
Q = {('s0','left'): 0.3, ('s0','right'): 0.1, ('s1','left'): 0.5, ('s1','right'): 0.7}
alpha = 0.2
r, s, a, s_next = 1.0, 's0', 'right', 's1'
td_target = r + gamma * max(Q[(s_next,'left')], Q[(s_next,'right')])
Q[(s,a)] += alpha * (td_target - Q[(s,a)])
print('Q(s0,right)=', round(Q[(s,a)], 6))

更新後の値が過去の値とどれだけ違うかは、学習率と TD 誤差で決まります。ここが調整ポイントです。



## 手を動かす 4: 探索と活用の切り替え

次に、探索率を変えたときの行動選択を見ます。探索不足は局所最適に閉じる典型的な原因です。


In [ ]:
import random

random.seed(7)


def choose_action(q_left, q_right, epsilon):
    greedy = 'left' if q_left >= q_right else 'right'
    if random.random() < epsilon:
        action = random.choice(['left', 'right'])
        return {'mode': 'explore', 'action': action, 'greedy': greedy}
    return {'mode': 'greedy', 'action': greedy, 'greedy': greedy}


for eps in [0.5, 0.1]:
    out = choose_action(0.4, 0.7, eps)
    print(f"epsilon={eps:.1f}", 'mode=', out['mode'], 'action=', out['action'], 'greedy=', out['greedy'])


探索率は固定せず、学習段階に応じて減衰させるのが一般的です。初期は広く探索し、後半で活用へ寄せます。



## 手を動かす 5: 方策評価の簡易チェック

最後に、方策の平均報酬を簡易的に比較します。アルゴリズムの評価は、更新式だけでなく結果の検証が不可欠です。


In [ ]:
episode_rewards = [1.2, 0.8, 1.5, 1.1, 1.4]
avg_reward = sum(episode_rewards) / len(episode_rewards)
variance = sum((r - avg_reward) ** 2 for r in episode_rewards) / len(episode_rewards)
print('avg =', round(avg_reward, 4))
print('var =', round(variance, 4))

平均だけでなく分散を見ると、方策の安定性も評価できます。実運用ではこの二軸が重要です。



## この節の要点

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 探索率が低すぎて行動が固定化する
2. 報酬設計が目的とずれている
3. 長期ロールアウトの不安定性を検証しない
